In [0]:
pip install langchain-google-genai python-dotenv tabulate delta-spark --quiet

In [0]:
%restart_python

In [0]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from pyspark.sql import SparkSession
import os

load_dotenv()

spark = SparkSession.builder.getOrCreate()

# lê os dois últimos registros da gold
df_gold = (
    spark
    .table('db_analytics.precos_semanal_bitcoin')
    .orderBy('semana_inicio', ascending=False)
    .limit(2)
    .toPandas()
)

dados_formatados = df_gold.to_markdown(index=False)

# lê o prompt do arquivo
with open('prompts/agente_analise.txt', 'r') as f:
    prompt_template = f.read()

# substitui o placeholder pelos dados
prompt = prompt_template.replace('{dados_semanal}', dados_formatados)

# chama o Gemini
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash-lite',
    google_api_key=os.getenv('GEMINI_API_KEY')
)

resposta = llm.invoke(prompt)
analise = resposta.content

print(analise)

In [0]:
# Para passar dados entre as tasks.
dbutils.jobs.taskValues.set(key='analise', value=analise)